# 09 — Revisao por LLM (Sabia 4) das Predicoes do Ensemble S2

**Objetivo:** Usar o modelo Sabia 4 (Maritaca AI) como revisor independente para auditar
os registros do ensemble S2 com `y_score > 0.05`, identificando potenciais
reclassificacoes (falsos negativos e falsos positivos).

**Estrategia:** Prompt **zero-shot** que define `mercado` como tema economico em
sentido amplo — mercado financeiro, atividade empresarial, indicadores macroeconomicos,
politica economica e fiscal, politicas trabalhistas e sociais com dimensao economica,
comercio, industria e agronegocio. Sem exemplos few-shot e sem ancoragem em criterio
editorial especifico.

**Entrada:** `artifacts/predictions_s2_factcheck_scrape_unified.csv`

**Saida:** `artifacts/revisao_sabia4_s2_factcheck_scrape_unified.csv`

In [2]:
import json
import os
from pathlib import Path

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

from economy_classifier.llm_review import (
    build_review_prompt,
    classify_batch,
    classify_single,
    compute_review_concordance,
)

/home/diacrono/Documentos/repositorios/economy-classifier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuracao

In [20]:
ARTIFACTS_DIR = Path("../artifacts")
INPUT_CSV = ARTIFACTS_DIR / "predictions_s2_standardized_records_combined.csv"
OUTPUT_CSV = ARTIFACTS_DIR / "revisao_sabia4_s2_standardized_records_combined.csv"
CHECKPOINT_CSV = ARTIFACTS_DIR / "revisao_sabia4_s2_checkpoint_standardized_records_combined.csv"

MODEL = "sabia-4"
BASE_URL = "https://chat.maritaca.ai/api"
TEMPERATURE = 0
SCORE_THRESHOLD = 0.05
CHECKPOINT_EVERY = 50

## 2. Carregar predicoes S2 e filtrar

In [21]:
full_df = pd.read_csv(INPUT_CSV, low_memory=False)
subset = full_df[full_df["y_score"] > SCORE_THRESHOLD].copy()
subset = subset.sort_values("record_id").reset_index(drop=True)

n_mercado = (subset["label"] == "mercado").sum()
n_outros = (subset["label"] == "outros").sum()

print(f"Total de registros no CSV: {len(full_df):,}")
print(f"Filtrados (y_score > {SCORE_THRESHOLD}): {len(subset):,}")
print(f"  mercado: {n_mercado:,}")
print(f"  outros:  {n_outros:,}")
subset[["record_id", "analysis_text", "label", "y_score", "score_bertimbau", "score_linearsvc"]].head(10)

Total de registros no CSV: 45,370
Filtrados (y_score > 0.05): 2,973
  mercado: 1,246
  outros:  1,727


,record_id,analysis_text,label,y_score,score_bertimbau,score_linearsvc
0,claimpt:news_0014_c2,"Particularmente, a monitorização tem de ser gl...",outros,0.0753,0.0067,0.8292
1,claimpt:news_0046_c1,Os resultados positivos do trabalho que tem vi...,outros,0.0518,0.0203,0.7388
2,claimpt:news_0046_c2,A nova infraestrutura é a base da nossa visão ...,outros,0.1390,0.9621,0.2613
3,claimpt:news_0100_c4,Apesar do aumento intercalar que nós conseguim...,outros,0.1059,0.9959,0.1742
4,claimpt:news_0125_c10,Tenho que no fundo continuar a assegurar no Po...,outros,0.0663,0.0056,0.8026
5,claimpt:news_0125_c2,"o que é importante é que na decisão, no anúnci...",outros,0.0864,0.3739,0.5882
6,claimpt:news_0125_c9,"é uma associação nacional, é uma associação qu...",outros,0.0952,0.0322,0.8619
7,factck_br:004ff5e7afeb1212,Boa parte do desemprego está nas cidades e no ...,outros,0.4586,0.9968,0.5688
8,factck_br:00629c4ca8aed0fb,Eu não tenho patrimônio.,outros,0.0591,0.0543,0.7417
9,factck_br:02f3fb35ede0d369,"Quando faltou gasolina, Parente estava na Petr...",mercado,0.6911,0.8954,0.8388


## 3. Prompt few-shot

6 exemplos fronteiricos que delimitam a fronteira entre "mercado" e "outros":

In [22]:
example_prompt = build_review_prompt("Texto de exemplo")
for msg in example_prompt:
    role = msg["role"].upper()
    content = msg["content"][:200]
    print(f"[{role}] {content}")
    print()

[SYSTEM] Voce e um classificador binario de textos jornalisticos brasileiros.

Sua tarefa: dado um trecho de texto, classifique-o segundo a sua tematica central como "mercado" (texto de tematica economica em s

[USER] Texto de exemplo



## 4. Inicializar cliente Maritaca AI

In [23]:
# REMOVIDO: chave hardcoded foi vazada e revogada. Defina MARITACA_API_KEY no ambiente antes de rodar.

In [24]:
client = OpenAI(
    api_key=os.environ["MARITACA_API_KEY"],
    base_url=BASE_URL,
    max_retries=3,
    timeout=30.0,
)

# Validacao rapida
test_result = classify_single(client, "Bolsa sobe 2% com alta do dolar", model=MODEL)
print(f"Teste de validacao: {test_result}")
assert test_result["label"] in ("mercado", "outros"), "API nao retornou label valido"
print("API funcionando.")

Teste de validacao: {'label': 'mercado', 'justificativa': 'O texto trata de movimentos do mercado financeiro, especificamente da bolsa e do dólar.', 'raw_response': '{"label": "mercado", "justificativa": "O texto trata de movimentos do mercado financeiro, especificamente da bolsa e do dólar."}'}
API funcionando.


## 5. Classificar com Sabia 4

Checkpoint a cada 50 registros para recuperacao em caso de interrupcao.

In [26]:
# Retomar de checkpoint se existir
if CHECKPOINT_CSV.exists():
    done_df = pd.read_csv(CHECKPOINT_CSV)
    done_ids = set(done_df["record_id"])
    remaining = subset[~subset["record_id"].isin(done_ids)]
    prior_results = done_df.to_dict("records")
    print(f"Retomando: {len(done_ids):,} ja classificados, {len(remaining):,} restantes")
else:
    remaining = subset
    prior_results = []
    print(f"Iniciando classificacao de {len(remaining):,} registros")

texts = remaining["analysis_text"].fillna("").tolist()
record_ids = remaining["record_id"].tolist()

pbar = tqdm(total=len(texts), desc="Sabia 4")
new_results = classify_batch(
    client, texts, record_ids,
    model=MODEL,
    temperature=TEMPERATURE,
    checkpoint_path=CHECKPOINT_CSV,
    checkpoint_every=CHECKPOINT_EVERY,
    progress_callback=pbar.update,
)
pbar.close()

all_results = prior_results + new_results
print(f"\nClassificacao concluida: {len(all_results):,} registros")

Retomando: 550 ja classificados, 2,423 restantes


Sabia 4: 100%|██████████| 2423/2423 [1:07:22<00:00,  1.67s/it]


Classificacao concluida: 2,973 registros


## 6. Montar DataFrame de resultados

In [27]:
results_df = pd.DataFrame(all_results)
results_df = results_df.rename(columns={
    "label": "label_sabia4",
    "justificativa": "justificativa_sabia4",
    "raw_response": "raw_response_sabia4",
})

output = subset.merge(results_df, on="record_id", how="left")

print(f"Registros no resultado final: {len(output):,}")
print(f"\nDistribuicao Sabia 4:")
print(output["label_sabia4"].value_counts())

Registros no resultado final: 2,973

Distribuicao Sabia 4:
label_sabia4
outros     1609
mercado    1296
erro         68
Name: count, dtype: int64


## 7. Salvar resultados

In [28]:
output.to_csv(OUTPUT_CSV, index=False)
print(f"Resultados salvos em {OUTPUT_CSV}")

# Limpar checkpoint
if CHECKPOINT_CSV.exists():
    CHECKPOINT_CSV.unlink()
    print("Checkpoint removido.")

Resultados salvos em ../artifacts/revisao_sabia4_s2_standardized_records_combined.csv
Checkpoint removido.


## 8. Analise de concordancia: Sabia 4 vs S2

In [29]:
conc = compute_review_concordance(output["label"], output["label_sabia4"])

print(f"Total avaliado:    {conc['total']:,}")
print(f"Erros de parsing:  {conc['erros_parsing']}")
print(f"Concordantes:      {conc['concordant']:,} ({conc['concordance_rate']:.2%})")
print(f"Discordantes:      {conc['discordant']:,}")
print(f"  outros -> mercado (falsos negativos S2): {conc['outros_to_mercado']}")
print(f"  mercado -> outros (falsos positivos S2): {conc['mercado_to_outros']}")
print(f"Cohen's Kappa:     {conc['cohen_kappa']:.4f}")

Total avaliado:    2,905
Erros de parsing:  68
Concordantes:      2,100 (72.29%)
Discordantes:      805
  outros -> mercado (falsos negativos S2): 449
  mercado -> outros (falsos positivos S2): 356
Cohen's Kappa:     0.4353


## 9. Tabela de contingencia

In [30]:
valid = output[output["label_sabia4"] != "erro"]
ct = pd.crosstab(valid["label"], valid["label_sabia4"], margins=True, margins_name="Total")
ct.index.name = "S2"
ct.columns.name = "Sabia 4"
ct

Sabia 4,mercado,outros,Total
S2,,,
mercado,847,356,1203
outros,449,1253,1702
Total,1296,1609,2905


## 10. Reclassificacoes detalhadas

### 10.1 Falsos negativos recuperados (S2=outros, Sabia=mercado)

In [31]:
fn_recovered = output[(output["label"] == "outros") & (output["label_sabia4"] == "mercado")]
print(f"Falsos negativos recuperados: {len(fn_recovered)}")
fn_recovered[["record_id", "analysis_text", "y_score", "score_bertimbau",
              "score_linearsvc", "justificativa_sabia4"]].head(20)

Falsos negativos recuperados: 449


,record_id,analysis_text,y_score,score_bertimbau,score_linearsvc,justificativa_sabia4
1,claimpt:news_0046_c1,Os resultados positivos do trabalho que tem vi...,0.0518,0.0203,0.7388,"O texto trata de resultados empresariais, conf..."
3,claimpt:news_0100_c4,Apesar do aumento intercalar que nós conseguim...,0.1059,0.9959,0.1742,"O texto aborda a perda de poder de compra, tem..."
5,claimpt:news_0125_c2,"o que é importante é que na decisão, no anúnci...",0.0864,0.3739,0.5882,O texto trata de planos para responder a neces...
7,factck_br:004ff5e7afeb1212,Boa parte do desemprego está nas cidades e no ...,0.4586,0.9968,0.5688,"O tema central é desemprego, um indicador macr..."
13,factck_br:069e982c0e4150ff,Idade mínima. Aposentadoria atual: não há; ...,0.0514,0.2178,0.5915,"O tema central é previdência, que possui dimen..."
15,factck_br:0d14182ec88b38d6,As pessoas em muitas cidades do Brasil deixam ...,0.1730,0.3366,0.7751,"O texto aborda a relação entre emprego formal,..."
18,factck_br:1327b1914ad13fb4,Neste momento em que temos um déficit foi apro...,0.0728,0.0050,0.8231,"O texto discute déficit orçamentário, desempre..."
19,factck_br:13610af614910f88,Bolsonaro alertou para reajuste de 37% na ener...,0.1133,0.0266,0.9049,O tema central envolve preços de energia e com...
23,factck_br:1b6b46d59f99bfcb,o Brasil era a oitava economia do mundo. Hoje ...,0.1631,0.3794,0.7293,O texto trata da posição do Brasil no ranking ...
25,factck_br:1c9e1340c133c2cc,55% das pessoas que estão na extrema pobreza e...,0.3514,0.7959,0.6273,"O tema central é pobreza, que possui dimensão ..."


### 10.2 Falsos positivos identificados (S2=mercado, Sabia=outros)

In [32]:
fp_identified = output[(output["label"] == "mercado") & (output["label_sabia4"] == "outros")]
print(f"Falsos positivos identificados: {len(fp_identified)}")
fp_identified[["record_id", "analysis_text", "y_score", "score_bertimbau",
               "score_linearsvc", "justificativa_sabia4"]].head(20)

Falsos positivos identificados: 356


,record_id,analysis_text,y_score,score_bertimbau,score_linearsvc,justificativa_sabia4
9,factck_br:02f3fb35ede0d369,"Quando faltou gasolina, Parente estava na Petr...",0.6911,0.8954,0.8388,O texto faz referência a um evento político-ad...
21,factck_br:14e81633653184fc,Ministério da Fazenda pede o fim da Uerj e dem...,0.8574,0.9949,0.9639,O tema central é uma proposta de reestruturaçã...
30,factck_br:2052ebb8cbd0221d,A que tem uma 'barrinha colorida' é produto qu...,0.6949,0.9474,0.8040,O texto trata de práticas de reaproveitamento ...
46,factck_br:31a8fc4edb925285,Valores gastos com publicidade nos governos de...,0.6293,0.9875,0.7154,O tema central é gasto público com publicidade...
61,factck_br:41ff88cf8049586e,Lula detém 52% das ações da &#39;Folha de S.Pa...,0.5117,0.9807,0.6235,O tema central é uma afirmação sobre proprieda...
78,factck_br:592d84deffeef32c,"Enquanto trabalhadores seguem para trabalhar, ...",0.8294,0.9950,0.9212,O trecho descreve uma cena cotidiana envolvend...
104,factck_br:75076c5c50dbda92,"Quando houve o apagão com FHC, Parente estava ...",0.7973,0.9719,0.8956,O tema central é um evento político-histórico ...
146,factck_br:b4489d90d114c1c8,"Após ouvir os argumentos dos trabalhadores, do...",0.7650,0.9988,0.8378,O tema central é uma decisão institucional da ...
175,factck_br:eb404db2b6cd23a8,'O Ibama leva dez anos para conceder uma licen...,0.6683,0.9717,0.7614,"O tema central é ambiental/regulatório, não ec..."
184,factck_br:f3d07f29594b3489,Temer minimiza desemprego com dados falsos - A...,0.8330,0.9977,0.9244,O tema central é uma crítica política ao uso d...


## 11. Erros de parsing

In [33]:
erros = output[output["label_sabia4"] == "erro"]
print(f"Erros de parsing: {len(erros)}")
if len(erros) > 0:
    erros[["record_id", "analysis_text", "raw_response_sabia4"]].head(10)

Erros de parsing: 68


## 12. Concordancia por faixa de score S2

In [34]:
valid = output[output["label_sabia4"] != "erro"].copy()
bins = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
labels = [f"{bins[i]:.2f}-{bins[i+1]:.2f}" for i in range(len(bins)-1)]
valid["faixa_score"] = pd.cut(valid["y_score"], bins=bins, labels=labels, include_lowest=True)

concorda = valid["label"] == valid["label_sabia4"]

summary = valid.groupby("faixa_score", observed=True).agg(
    total=("record_id", "count"),
    concordantes=("label", lambda x: (x == valid.loc[x.index, "label_sabia4"]).sum()),
).reset_index()
summary["taxa_concordancia"] = (summary["concordantes"] / summary["total"]).round(4)
summary

,faixa_score,total,concordantes,taxa_concordancia
0,0.05-0.10,750,638,0.8507
1,0.10-0.20,422,293,0.6943
2,0.20-0.30,193,130,0.6736
3,0.30-0.40,171,103,0.6023
4,0.40-0.50,166,89,0.5361
5,0.50-0.60,142,77,0.5423
6,0.60-0.70,209,100,0.4785
7,0.70-0.80,292,197,0.6747
8,0.80-0.90,560,473,0.8446


## 13. Distribuicao de scores nos registros reclassificados

In [35]:
reclassified = output[output["label"] != output["label_sabia4"]]
reclassified = reclassified[reclassified["label_sabia4"] != "erro"]

if len(reclassified) > 0:
    print(f"Registros reclassificados: {len(reclassified)}")
    print(f"\ny_score:")
    print(reclassified["y_score"].describe().round(4))
    print(f"\nscore_bertimbau:")
    print(reclassified["score_bertimbau"].describe().round(4))
    print(f"\nscore_linearsvc:")
    print(reclassified["score_linearsvc"].describe().round(4))
else:
    print("Nenhum registro reclassificado.")

Registros reclassificados: 805

y_score:
count    805.0000
mean       0.4376
std        0.2696
min        0.0501
25%        0.1631
50%        0.4323
75%        0.6881
max        0.8764
Name: y_score, dtype: float64

score_bertimbau:
count    805.0000
mean       0.8113
std        0.3251
min        0.0002
25%        0.8327
50%        0.9836
75%        0.9961
max        0.9996
Name: score_bertimbau, dtype: float64

score_linearsvc:
count    805.0000
mean       0.6608
std        0.2220
min        0.0282
25%        0.5247
50%        0.7072
75%        0.8361
max        0.9973
Name: score_linearsvc, dtype: float64
